In [2]:
from worker.batch import get_portfolio_positions
from worker.database import connect
import datetime

import polars as pl
from scipy import stats

import math
import time

In [3]:
from worker.database import Measure
from worker.utils import insert_list_of_dicts


ref_date = datetime.date(2025, 8, 29)
first_calc_date = datetime.date(2024, 9, 1)
first_price_date = datetime.date(2016, 1, 1)

s_price_dates = pl.date_range(first_price_date, ref_date, eager=True)


def _single_loop(df_positions: pl.DataFrame, df_market_data: pl.DataFrame, ptf_id: str, calc_date: datetime.date):
    df_ = (df_positions.filter(pl.col("portfolio_id").eq(ptf_id)).filter(pl.col("date").eq(calc_date))).drop(
        "portfolio_id", "date"
    )

    df_prices = df_.select("instrument_id").join(df_market_data, how="right", on=["instrument_id"])

    df_ = (
        df_
        .join(df_prices, on=["instrument_id"])
        .filter(pl.col("date") <= calc_date)
        .with_columns(value=pl.col("quantity") * pl.col("price"))
        .group_by("date")
        .agg(value=pl.col("value").sum())
        .sort("date")
        .with_columns(returns=pl.col("value").pct_change())
    )

    rets = df_["returns"]
    value = df_["value"][-1]

    assert len(rets) > 60

    vol = rets.tail(60).std()

    assert isinstance(vol, float)

    annualized_vol = vol * math.sqrt(250)
    longer_vol = rets.tail(250).std()

    hist_var = rets.tail(500).quantile(1 - 0.99)
    assert hist_var is not None

    hist_var = - hist_var

    ewma_backcast_window = 60
    ewma_backcast_vol = rets[:ewma_backcast_window].std()
    curr = ewma_backcast_vol
    ewma_vols = [curr]

    for ret in rets[ewma_backcast_window:]:
        assert isinstance(curr, float)
        curr = math.sqrt(0.94 * curr **2 + 0.06 * ret**2)
        ewma_vols.append(curr)

    ewma_vol = ewma_vols[-1]
    assert isinstance(ewma_vol, float)
    assert isinstance(longer_vol, float)

    ewma_var = float(-ewma_vol * stats.norm.ppf(1 - 0.99))
    param_var = float(-longer_vol * stats.norm.ppf(1 - 0.99))

    # Put it all together
    return pl.DataFrame(
        {
            "portfolio_id": ptf_id,
            "calc_date": calc_date,
            "ptf_value": value,
            "hist_var": hist_var,
            "ewma_var": ewma_var,
            "param_var": param_var,
            "ex_ante_vol": annualized_vol,
        }
    )    


with connect() as session:
    # Load market data and forward fill
    df_market_data = (
        pl.read_database(
            """
            SELECT
                date,
                instrument_id::TEXT as instrument_id,
                value AS price
            FROM market_data
            WHERE date <= :date
            AND data_type = 'adj_close'""",
            session,
            execute_options={"params": {"date": ref_date}},
        )
        .group_by("instrument_id")
        .map_groups(lambda df: (df.sort("date").with_columns(price=pl.col("price").forward_fill())))
    )

    # Get positions for all calculation dates
    df_positions = get_portfolio_positions(session, ref_date).filter(pl.col("date") >= first_calc_date)

    # For each portfolio and date, we run the calculation loop
    combinations = df_positions.select("portfolio_id", "date").unique()

    dfs = []
    for i, (ptf_id, calc_date) in enumerate(combinations.rows()):
        print(f"{i} / {len(combinations)}")
        dfs.append(_single_loop(df_positions, df_market_data, ptf_id, calc_date))

    df_measures = (
        pl.concat(dfs)
        .unpivot(
            index=["portfolio_id", "calc_date"]
        )
        .rename({"variable": "measure", "calc_date": "date"})
    )


    insert_list_of_dicts(items=df_measures.to_dicts(), table=Measure, session=session)
        


0 / 13000
1 / 13000
2 / 13000
3 / 13000
4 / 13000
5 / 13000
6 / 13000
7 / 13000
8 / 13000
9 / 13000
10 / 13000
11 / 13000
12 / 13000
13 / 13000
14 / 13000
15 / 13000
16 / 13000
17 / 13000
18 / 13000
19 / 13000
20 / 13000
21 / 13000
22 / 13000
23 / 13000
24 / 13000
25 / 13000
26 / 13000
27 / 13000
28 / 13000
29 / 13000
30 / 13000
31 / 13000
32 / 13000
33 / 13000
34 / 13000
35 / 13000
36 / 13000
37 / 13000
38 / 13000
39 / 13000
40 / 13000
41 / 13000
42 / 13000
43 / 13000
44 / 13000
45 / 13000
46 / 13000
47 / 13000
48 / 13000
49 / 13000
50 / 13000
51 / 13000
52 / 13000
53 / 13000
54 / 13000
55 / 13000
56 / 13000
57 / 13000
58 / 13000
59 / 13000
60 / 13000
61 / 13000
62 / 13000
63 / 13000
64 / 13000
65 / 13000
66 / 13000
67 / 13000
68 / 13000
69 / 13000
70 / 13000
71 / 13000
72 / 13000
73 / 13000
74 / 13000
75 / 13000
76 / 13000
77 / 13000
78 / 13000
79 / 13000
80 / 13000
81 / 13000
82 / 13000
83 / 13000
84 / 13000
85 / 13000
86 / 13000
87 / 13000
88 / 13000
89 / 13000
90 / 13000
91 / 1300

In [3]:


df_measures

portfolio_id,date,measure,value
str,date,str,f64
"""0b209d5c-12ed-48af-a597-b1e22f…",2024-11-08,"""ptf_value""",44650.0
"""0a501045-acfb-41c3-ba19-e64d56…",2025-05-26,"""ptf_value""",59590.0
"""e3a50d76-dcb7-4377-8ad3-8ff9f4…",2025-07-11,"""ptf_value""",25470.0
"""ca366c28-aff3-45f6-ae3b-37079c…",2025-01-24,"""ptf_value""",415490.0
"""78e37b84-0a65-4f70-a706-ae0175…",2025-07-22,"""ptf_value""",112830.0
…,…,…,…
"""da0955b6-52f2-4c70-bd60-e3a379…",2024-10-11,"""ex_ante_vol""",0.206994
"""64ae58df-139d-4425-a307-05072e…",2024-11-06,"""ex_ante_vol""",0.127289
"""00e01b8c-84f4-4a73-9345-ed2ad7…",2025-02-13,"""ex_ante_vol""",0.126502
